# TP6 — Detección de Postura Corporal con MediaPipe
**Instituto de Formación Técnica Superior N° 33**  
**Materia:** Técnicas de Procesamiento de Imágenes

---

## Paso 1 — Instalar dependencias

In [ ]:
!pip install mediapipe -q
print('✅ Instalación completa')

## Paso 2 — Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

## Paso 3 — Cargar y mostrar la imagen

In [ ]:
import cv2
import matplotlib.pyplot as plt

ruta = '/content/drive/MyDrive/Procesamiento-Imagenes/TP6/postura.jpg'

img = cv2.imread(ruta)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))
plt.imshow(img_rgb)
plt.title('Imagen original — Buena y mala postura')
plt.axis('off')
plt.show()

## Paso 4 — Detectar landmarks con MediaPipe

In [ ]:
import mediapipe as mp
import cv2
import matplotlib.pyplot as plt

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# Procesar imagen
with mp_pose.Pose(static_image_mode=True, model_complexity=2,
                  min_detection_confidence=0.5) as pose:

    img_rgb2 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    resultados = pose.process(img_rgb2)

    # Dibujar landmarks
    img_landmarks = img.copy()
    if resultados.pose_landmarks:
        mp_drawing.draw_landmarks(
            img_landmarks,
            resultados.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )
        print('✅ Landmarks detectados')
    else:
        print('⚠️ No se detectaron landmarks')

img_landmarks_rgb = cv2.cvtColor(img_landmarks, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 6))
plt.imshow(img_landmarks_rgb)
plt.title('Landmarks detectados por MediaPipe')
plt.axis('off')
plt.show()

## Paso 5 — Calcular ángulos de postura

In [ ]:
import numpy as np

def calcular_angulo(p1, p2):
    """Calcula el ángulo de inclinación respecto al eje vertical."""
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    angulo = np.degrees(np.arctan2(dx, dy))
    return abs(angulo)

if resultados.pose_landmarks:
    landmarks = resultados.pose_landmarks.landmark
    h, w, _ = img.shape

    # Puntos clave lado izquierdo
    oreja   = (landmarks[mp_pose.PoseLandmark.LEFT_EAR].x * w,
               landmarks[mp_pose.PoseLandmark.LEFT_EAR].y * h)
    hombro  = (landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER].x * w,
               landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER].y * h)
    cadera  = (landmarks[mp_pose.PoseLandmark.LEFT_HIP].x * w,
               landmarks[mp_pose.PoseLandmark.LEFT_HIP].y * h)

    ang_cuello = calcular_angulo(hombro, oreja)
    ang_torso  = calcular_angulo(cadera, hombro)

    print(f'Inclinación del cuello: {ang_cuello:.1f}°')
    print(f'Inclinación del torso:  {ang_torso:.1f}°')

    if ang_cuello < 40 and ang_torso < 10:
        print('✅ POSTURA CORRECTA')
    else:
        print('⚠️ POSTURA INCORRECTA — corregir posición')

## Paso 6 — Comparativa original vs landmarks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(img_rgb)
axes[0].set_title('Imagen original')
axes[0].axis('off')
axes[1].imshow(img_landmarks_rgb)
axes[1].set_title('Detección MediaPipe')
axes[1].axis('off')
plt.tight_layout()
plt.show()

---
*IFTS N° 33 — Técnicas de Procesamiento de Imágenes*